# 1.3 dalis (a) — „TensorFlow“ neuroninis tinklas
### Kliento užklausos ketinimo klasifikavimas `„AdventureWorks DW using TensorFlow“ Colab stiliuje`

Ši užrašinė (angl. *notebook*) — kurso 1.3 dalis: originalus „AdventureWorks data warehouse“
(TF) „Colab“ dokumentas buvo **suprastas ir perdarytas su šio projekto duomenimis**.

**Kaip veikia originalus „Colab“ (AdventureWorks DW + TensorFlow):**
1. Iš duomenų saugyklos (angl. *data warehouse*, DW) imami struktūrizuoti klientų / pardavimų įrašai;
2. Požymiai (X) paruošiami ir normalizuojami;
3. „TensorFlow“ `Dense` sluoksnių neuroninis tinklas mokomas spręsti
   klasifikavimo užduotį (tikslinė klasė y);
4. Vertinamas tikslumu, mokymosi kreivėmis ir painiavos matrica.

**Mūsų atitikmuo:** vietoj „AdventureWorks DW“ lentelių naudojame šio projekto
„duomenų saugyklą“ — `data/customer_support_dataset.csv` (252 (X, y) poros):
- **X** = kliento užklausos tekstas → TF-IDF požymiai (simbolių *n*-gramos, kaip 1.2.2 dalyje);
- **y** = ketinimo klasė iš 6 kategorijų (`order_status`, `delivery`, `returns`, `payment`,
  `warranty`, `product_usage`);
- Modelis — tankusis (Dense / MLP) tinklas su `Dropout` reguliarizacija.

GPU nebūtinas — modelis nedidelis, pakanka CPU.


## 1. Bibliotekos ir atsitiktinumo valdymas
Fiksuojame `SEED`, kad rezultatai būtų atkuriami.

In [ ]:
%pip install -q tensorflow scikit-learn pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

import tensorflow as tf
tf.random.set_seed(SEED)
print('TensorFlow version:', tf.__version__)

## 2. Duomenys: klientų užklausų rinkinys (X, y)
Generatorius yra deterministinis (toks pat kaip ir `scripts/generate_dataset.py`),
todėl užrašinėje gauti duomenys visada identiški `.csv` faile esantiems —
(X, y) poros niekada „neišsiskiria“.

In [ ]:
# --- Deterministic dataset generator (same as scripts/generate_dataset.py) ---
# X = customer query text, y = intent label (6 classes).
DATA_SPEC = {
    'order_status': [
        ('Kur mano užsakymas {order}?', 'order'),
        ('Kokia yra užsakymo {order} būsena?', 'order'),
        ('Nematau užsakymo {order} paskyroje, ką daryti?', 'order'),
        ('Kaip sužinoti mano užsakymo {order} statusą?', 'order'),
        ('Ar užsakymas {order} jau išsiųstas?', 'order'),
        ('Mano užsakymo {order} statusas vis dar „Apmokėta“, ar tai normalu?', 'order'),
        ('Kada bus matoma užsakymo {order} sekimo informacija?', 'order'),
    ],
    'delivery': [
        ('Kiek trunka pristatymas į {city}?', 'city'),
        ('Ar pristatote siuntas į {city} savaitgaliais?', 'city'),
        ('Kokia pristatymo į {city} kaina?', 'city'),
        ('Kada mano siunta pasieks {city}?', 'city'),
        ('Ar taikomas nemokamas pristatymas perkant už {eur} eurų?', 'eur'),
        ('Kur rasti siuntos į {city} sekimo numerį?', 'city'),
        ('Ar galiu atsiimti siuntą į {city} paštomate?', 'city'),
    ],
    'returns': [
        ('Noriu grąžinti {item}, kaip tai padaryti?', 'item'),
        ('Ar galiu grąžinti prekę po {days} dienų?', 'days'),
        ('Kiek kainuoja grąžinti {item}?', 'item'),
        ('Kada atgausiu pinigus grąžinęs {item}?', 'item'),
        ('Ar galima grąžinti {item} originalioje pakuotėje?', 'item'),
        ('Kur rasiu grąžinimo formą norėdamas grąžinti {item}?', 'item'),
        ('Ar galima grąžinti {item}, jei pakuotė jau buvo atidaryta?', 'item'),
    ],
    'payment': [
        ('Ar priimate {method}?', 'method'),
        ('Kiek laiko trunka mokėjimo autorizacija pasirinkus {method}?', 'method'),
        ('Nepavyko apmokėti užsakymo pasirinkus {method}, ką daryti?', 'method'),
        ('Ar galima mokėti dalimis pasirinkus {method}?', 'method'),
        ('Kada gausiu sąskaitą faktūrą apmokėjus per {method}?', 'method'),
        ('Ar palaikote {method} internetinėje parduotuvėje?', 'method'),
        ('Ką daryti, jei mokant per {method} nurašė pinigus, bet užsakymas neapmokėtas?', 'method'),
    ],
    'warranty': [
        ('Kiek laiko galioja garantija įsigijus {item}?', 'item'),
        ('Ar nemokamai suremontuosite {item} pagal garantiją?', 'item'),
        ('Ką daryti, jei gaminys sugedo po {days} dienų naudojimo?', 'days'),
        ('Ar reikia atskiro talono pateikiant {item} garantiniam remontui?', 'item'),
        ('Kiek trunka garantinis remontas pateikus {item}?', 'item'),
        ('Ar garantija galioja, jei netyčia pažeidžiau {item}?', 'item'),
        ('Ar atvežate {item} į remontą nemokamai?', 'item'),
    ],
    'product_usage': [
        ('Kaip susieti {item} su telefonu?', 'item'),
        ('Kur rasti naudotojo vadovą įsigijus {item}?', 'item'),
        ('Nusipirkau {item}, bet prietaisas neįsijungia, ką daryti?', 'item'),
        ('Kaip atstatyti gamyklinius nustatymus naudojant {item}?', 'item'),
        ('Ar galima gauti naudotojo vadovą lietuvių kalba įsigijus {item}?', 'item'),
        ('Kaip tinkamai prijungti {item} prie kompiuterio?', 'item'),
        ('Kiek laiko reikia krauti {item}?', 'item'),
    ],
}

SLOT_VALUES = {
    'order': ['TS-102345', 'TS-201111', 'TS-314560', 'TS-456789', 'TS-511235', 'TS-623456'],
    'city': ['Vilnių', 'Kauną', 'Klaipėdą', 'Šiaulius', 'Panevėžį', 'Alytų'],
    'item': ['išmanųjį telefoną', 'belaides ausines', 'išorinę bateriją',
             'elektroninę skaityklę', 'žaidimų pultelį', 'internetinę kamerą'],
    'days': ['3', '7', '10', '13', '20', '30'],
    'eur': ['30', '45', '60', '75', '90', '120'],
    'method': ['PayPal', 'Visa kortelę', 'Mastercard kortelę',
               'bankinį pavedimą', 'Google Pay', 'Apple Pay'],
}

INTENTS = ['order_status', 'delivery', 'returns', 'payment', 'warranty', 'product_usage']


def generate_dataset():
    rows = []
    for intent, specs in DATA_SPEC.items():
        for template, slot in specs:
            for value in SLOT_VALUES[slot]:
                rows.append((template.replace('{' + slot + '}', value), intent))
    return pd.DataFrame(rows, columns=['query', 'intent'])


df = generate_dataset()
print(f'Dataset: {len(df)} (X, y) pairs, {len(INTENTS)} classes')
print(df['intent'].value_counts())

# Canonical hold-out test set (same 6 queries as data/customer_support_test.csv
# and as the 1.2.2 Gemini prototype) - used to compare all models fairly.
TEST_EXAMPLES = [
    ('Kokiomis dienomis pristatote užsakymus į Vilnių?', 'delivery'),
    ('Norėčiau grąžinti prekę po 10 dienų nuo pirkimo, ar įmanoma?', 'returns'),
    ('Nematau savo užsakymo istorijos paskyroje, ką daryti?', 'order_status'),
    ('Ar parduotuvė priima PayPal mokėjimus?', 'payment'),
    ('Ar už 25 eurus pirktas gaminys turi garantiją?', 'warranty'),
    ('Kur rasti ausinių susiejimo instrukciją?', 'product_usage'),
]

## 3. Požymių inžinerija: tekstas → TF-IDF vektoriai
Naudojame **simbolių *n*-gramas** (`char_wb`, 3–5) — dėl lietuvių kalbos
linksniavimo žodžių lygio žodynas prasčiau aprėpia testavimo užklausas.
TF-IDF reikšmes paverčiame į tankų `float32` masyvą, o etiketes užkoduojame sveikaisiais skaičiais.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 80 % mokymui / 20 % validacijai; kanoninė 6 užklausų aibė - galutiniam testui
X_train_text, X_val_text, y_train_raw, y_val_raw = train_test_split(
    df['query'], df['intent'], test_size=0.2, random_state=SEED, stratify=df['intent'])

vectorizer = TfidfVectorizer(lowercase=True, analyzer='char_wb', ngram_range=(3, 5))
X_train = vectorizer.fit_transform(X_train_text).toarray().astype('float32')
X_val = vectorizer.transform(X_val_text).toarray().astype('float32')
X_test = vectorizer.transform([q for q, _ in TEST_EXAMPLES]).toarray().astype('float32')

label_to_id = {label: i for i, label in enumerate(INTENTS)}
y_train = np.array([label_to_id[l] for l in y_train_raw])
y_val = np.array([label_to_id[l] for l in y_val_raw])
y_test = np.array([label_to_id[l] for _, l in TEST_EXAMPLES])

print(f'X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')
print(f'Požymių (TF-IDF) skaičius: {X_train.shape[1]}')

## 4. Modelio architektūra (Dense / MLP)

Nuoseklusis (`Sequential`) modelis, analogiškas originalo „Colab“ struktūrai:

| Sluoksnis | Paskirtis |
|---|---|
| `InputLayer` (*n* požymių) | TF-IDF vektoriaus įvestis |
| `Dense(64, relu)` | pirmoji netiesinė kombinacija |
| `Dropout(0.3)` | reguliarizacija prieš persimokymą (angl. *overfitting*) |
| `Dense(32, relu)` | gilesnė reprezentacija |
| `Dense(6, softmax)` | tikimybės 6 ketinimo klasėms |

`SparseCategoricalCrossentropy` — etiketės yra sveikieji skaičiai (ne *one-hot*).
Optimizatorius `Adam` — adaptyvus mokymosi greitis, standartinis pasirinkimas.


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(len(INTENTS), activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## 5. Mokymasis
`EarlyStopping` stebi validacijos tikslumą ir anksti sustabdo mokymą,
kai modelis nustoja tobulėti — taip išvengiama persimokymo.

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100, batch_size=16,
    callbacks=[early_stop],
    verbose=2,
)

In [ ]:
# --- Mokymosi kreivės ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='mokymo nuostoliai')
axes[0].plot(history.history['val_loss'], label='validacijos nuostoliai')
axes[0].set_xlabel('Epocha'); axes[0].set_ylabel('Nuostoliai (Loss)'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history.history['accuracy'], label='mokymo tikslumas')
axes[1].plot(history.history['val_accuracy'], label='validacijos tikslumas')
axes[1].set_xlabel('Epocha'); axes[1].set_ylabel('Tikslumas (Accuracy)'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('TensorFlow MLP mokymosi kreivės')
plt.tight_layout(); plt.show()

val_acc = max(history.history['val_accuracy'])
print(f'Geriausias validacijos tikslumas: {val_acc:.4f}')

## 6. Vertinimas: kanoninė testavimo aibė ir painiavos matrica
Tą pačią 6 užklausų aibę naudos **visi** 1.3–1.5 modeliai — tai leidžia
objektyviai palyginti MLP, sprendimų miškus, CNN ir perkeltąjį mokymą (angl. *transfer learning*).

Painiavos matricos braižymo pagalbinė funkcija pateikiama žemiau.

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes, title):
    cm = np.zeros((len(classes), len(classes)), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    fig, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(classes)), classes, rotation=45, ha='right')
    ax.set_yticks(range(len(classes)), classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_xlabel('Numatytas ketinimas')
    ax.set_ylabel('Tikrasis ketinimas')
    ax.set_title(title)
    fig.colorbar(im)
    plt.tight_layout()
    plt.show()
    return cm

val_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
val_accuracy = accuracy_score(y_val, val_pred)

test_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
test_accuracy = accuracy_score(y_test, test_pred)

print(f'Validacijos tikslumas: {val_accuracy:.4f}')
print(f'Testavimo (6 užklausos) tikslumas: {test_accuracy:.4f}')
print()
for (query, true_label), pred_id in zip(TEST_EXAMPLES, test_pred):
    mark = 'OK ' if INTENTS[pred_id] == true_label else 'WRONG'
    print(f'{mark} true={true_label:14s} pred={INTENTS[pred_id]:14s} {query}')

cm = plot_confusion_matrix(y_val, val_pred, INTENTS, 'MLP — validacijos painiavos matrica')
print(classification_report(y_val, val_pred, target_names=INTENTS))

## 7. Išvados
1. Tekstinės užklausos sėkmingai paverčiamos požymiais (TF-IDF, simbolių *n*-gramos)
   ir klasifikuojamos tankiuoju tinklu — originalaus „AdventureWorks Colab“
   „DW lentelė → Dense tinklas“ schema pritaikyta tekstiniams duomenims.
2. Reguliarių šablonų duomenimis MLP pasiekia aukštą validacijos tikslumą;
   klaidos (jei pasitaiko) atsiranda tarp semantiškai artimų klasių.
3. Palyginimui 1.3(b) dalyje tas pats uždavinys sprendžiamas
   **„TensorFlow Decision Forests“** modeliais be rankinės požymių inžinerijos.
